<font color="green"><font size="6">**INTERFACE**</font></font>

In [1]:
# Interface with tkinter

# Import libraries
import pandas as pd
import numpy as np
from tkinter import *
from tkinter import ttk
from tkinter import filedialog
from tkinter import messagebox, simpledialog
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from PIL import Image, ImageTk

import platform

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score



# Global variable: program version
PROGRAM_VERSION = "1.0"


def load_data():
    """
        This function loads a CSV file and returns a Pandas DataFrame via a dialog box
        to allow the user to select the file to upload.

        :arg:()

        :return: df = loaded dataset
    """
    # File types that can be selected in the dialog box
    filetypes=(
        ('Text files','*.csv'), # * : any name
        ('All files','*.*')
        )
    filename = filedialog.askopenfilename(title = "Open File", initialdir="/",filetypes=filetypes) 
    # initialdir="/" : initial directory = C

    if filename:
        try:
            df = pd.read_csv(filename) # read the file
            messagebox.showinfo("Success", "Dataset loaded successfully!")
            return df
        except Exception as e:
            messagebox.showerror("Error", f"Error loading file: {e}")

    return None # executed if no file is selected



# Interface class
class Datawindow:
    # Initialize class members
    def __init__(self, root):
        self.root = root
        
        self.df = pd.DataFrame()  
        
        # Variables for saving
        self.current_plot_image = None 
        self.current_model_results = None

        # Creation of the PanedWindow for resizing panels by dragging the separator bars horizontally
        self.paned_window = PanedWindow(root, orient=HORIZONTAL)
        self.paned_window.pack(fill=BOTH, expand=True)

        # Frame for the menu 
        self.menu_frame = Frame(self.paned_window, bg="RoyalBlue", width=200)
        self.paned_window.add(self.menu_frame, stretch="always")

        # Notebook for content
        self.notebook = ttk.Notebook(self.paned_window)
        self.paned_window.add(self.notebook, stretch="always")

        # Menubar: main menu bar of a window
        self.menubar = Menu(self.root)
        self.root.config(menu=self.menubar)

        file_menu = Menu(self.menubar, tearoff=0)
        self.menubar.add_cascade(label="File", menu=file_menu)
        file_menu.add_command(label="Load Dataset", command=self.load_dataset)
        file_menu.add_command(label="Save", command=self.save_content)
        file_menu.add_separator()
        file_menu.add_command(label="Exit", command=root.destroy)

        # Buttons for the list of menu options
        self.home_button = Button(self.menu_frame, text="Home", command=self.show_home)
        self.home_button.pack(fill='x')
        
        self.view_button = Button(self.menu_frame, text="View Dataset", command=self.view_data)
        self.view_button.pack(fill='x')

        # "Graphs" button with drop-down menu
        self.graph_button = Menubutton(self.menu_frame, text="Graphs", relief=RAISED)
        self.graph_menu = Menu(self.graph_button, tearoff=0)
        self.graph_button.config(menu=self.graph_menu)
        
        self.graph_menu.add_command(label="Countplot", command=lambda: self.plot_data('countplot'))
        self.graph_menu.add_command(label="Boxplot", command=lambda: self.plot_data('box'))
        self.graph_menu.add_command(label="Line plot", command=lambda: self.plot_data('line'))
        self.graph_menu.add_command(label="Load graph", command=self.load_image)
        self.graph_button.pack(fill = 'x')

        # "ML Models" button with drop-down menu
        self.ml_button = Menubutton(self.menu_frame, text="ML Models",  relief=RAISED)
        self.ml_menu = Menu(self.ml_button, tearoff=0)
        self.ml_button.config(menu=self.ml_menu)
        
        self.ml_menu.add_command(label="Logistic Regression", command=lambda: self.apply_model('logistic_regression'))
        self.ml_menu.add_command(label="Decision Tree", command=lambda: self.apply_model('decision_tree'))
        self.ml_menu.add_command(label="Random Forest", command=lambda: self.apply_model('random_forest'))
        self.ml_menu.add_command(label="SVC", command=lambda: self.apply_model('SVC'))
        self.ml_menu.add_command(label="MLP", command=lambda: self.apply_model('MLP'))
        self.ml_button.pack(fill = 'x')
        
        # Show the home screen on opening
        self.show_home()  


    def load_dataset(self):
        """
            Method that calls the load_data() function to load the Dataset.
            It is called when the "Load Dataset" command is selected.

            :arg:()
        """
        self.df = load_data()


    def view_data(self):
        """
            Method for displaying dataset data within the interface.
            It is called when the "View Dataset" button is pressed.

            :arg:()
        """
        if self.df is not None and not self.df.empty:
            print("Displaying the dataset")
    
            # Check if a data visualization tab already exists
            for tab in self.notebook.tabs():
                if self.notebook.tab(tab, "text") == "View Dataset":
                    self.notebook.select(tab) # if it exists, it is selected instead of creating a new one
                    return
            
            # Creating the Treeview (widget that provides a tabular view of data) and scrollbars
            tree_frame = Frame(self.notebook)
            tree_scroll_y = Scrollbar(tree_frame, orient="vertical")
            tree_scroll_y.pack(side="right", fill="y")

            tree_scroll_x = Scrollbar(tree_frame, orient="horizontal")
            tree_scroll_x.pack(side="bottom", fill="x")

            tree = ttk.Treeview(tree_frame, yscrollcommand=tree_scroll_y.set, xscrollcommand=tree_scroll_x.set)
            tree.pack(fill='both', expand=True)

            tree_scroll_y.config(command=tree.yview)
            tree_scroll_x.config(command=tree.xview)
            
            # Setting the columns
            tree["columns"] = list(self.df.columns)
            tree["show"] = "headings" # only the column headings should be displayed

            for column in tree["columns"]:
                tree.heading(column, text=column)
                tree.column(column, width=100, anchor='center')

            # Iterate over each row of the DataFrame
            for index, row in self.df.iterrows(): 
                tree.insert("", "end", values=list(row)) # end: the new line must be added to the end of the Treeview
            
            # Adding the card to the notebook
            self.add_tab(tree_frame, "View Dataset")
        else:
            # If the dataset is not loaded or is empty, show an error message
            messagebox.showerror("Error", "No dataset loaded")


    def plot_data(self, chart_type):
        """
            Method that creates different types of graphs (countplot, boxplot, line plot) based on the dataset
            data and displays them in a notebook tab.
            It is called when "Countplot", "Boxplot", "Line plot" commands of the "Graphs" Menu are selected.

            :arg: chart_type = variable to define the type of graph to plot
        """
        if self.df is not None and not self.df.empty:
            # Title of the card
            title = f"{chart_type.capitalize()} Graph"
            
            # Check if a tab already exists for this type of chart
            for tab in self.notebook.tabs():
                if self.notebook.tab(tab, "text").startswith(title):
                    self.notebook.select(tab)
                    return
            
            # Creation of the graph figure
            figure = plt.Figure(figsize=(6, 5), dpi=100)
            ax = figure.add_subplot(111)
            # 111 : number of rows, number of columns and index of the subplot
        
            # Creating the chart based on type:
            if chart_type == 'countplot':
                # Prompts the user to enter the name of the target variable
                target_column = simpledialog.askstring("Input", "Enter the name of the target variable:")

                if target_column not in self.df.columns:
                    messagebox.showerror("Error", "The target variable is not present in the dataset.")
                    return
                
                sns.countplot(x=target_column, data=self.df, ax=ax)
                ax.set_title(f"Countplot of {target_column}")
               
            elif chart_type == 'box':
                target_column = simpledialog.askstring("Input", "Enter the name of the target variable:")

                if target_column not in self.df.columns:
                    messagebox.showerror("Error", "The target variable is not present in the dataset.")
                    return
                
                # New DataFrame without the target variable
                df_notarg = pd.DataFrame(self.df.drop(target_column, axis=1))

                figure, axs = plt.subplots(nrows=5, ncols=5, figsize=(15, 15))
                axs = axs.flatten()
                # Define a color palette using Seaborn
                colors = sns.color_palette("husl", n_colors=len(df_notarg.columns))

                for index, (k, v) in enumerate(df_notarg.items()):
                    sns.boxplot(y=k, data=df_notarg, ax=axs[index], color=colors[index % len(colors)])

                # Removing unused axes
                for j in range(index+1, len(axs)):
                    axs[j].axis('off')

                plt.tight_layout() # ensures that subplots do not overlap

            elif chart_type == 'line':
                # Prompts the user to enter the name of a variable in the dataset to be plotted
                column = simpledialog.askstring("Input", "Enter the name of the variable to analyze:")

                if column not in self.df.columns:
                    messagebox.showerror("Error", "The variable is not present in the dataset.")
                    return
                
                subset = self.df[column][::10] # gets every 10 points for better visualization of large datasets
                ax.plot(subset, marker='o', linestyle='-', linewidth=0.5, color='b', alpha = 0.7)
                ax.set_title(f'Line Plot of {column}')
                ax.set_xlabel('Index')
                ax.set_ylabel(column)
                ax.grid(True)
            
            # Converting the Matplotlib figure into a Tkinter widget
            canvas = FigureCanvasTkAgg(figure, self.notebook)
            # Adds the chart widget to the Tkinter notebook and resizes it to fill all available space
            canvas.get_tk_widget().pack(fill='both', expand=True)

            self.add_tab(canvas.get_tk_widget(), title)

            # Memorize the graph image for saving
            self.current_plot_image = figure
        else:
            messagebox.showerror("Error", "No dataset loaded")


    def load_image(self):
        """
            This method allows the user to select an image file from their system, upload it, resize it
            and display it in a new tab of a graphical interface.
            It is called when "Load graph" command of the "Graphs" Menu is selected.

            :arg:()
        """
        # Opening a dialog box to select a file
        file_path = filedialog.askopenfilename(
            title="File",
            initialdir="/",
            filetypes=(("PNG files", "*.png"), ("JPEG files", "*.jpg;*.jpeg"), ("All files", "*.*"))
        )

        if file_path: # if a file has been selected
            try:
                image = Image.open(file_path)
                image = image.resize((400, 400), Image.Resampling.LANCZOS)  # the image is resized
                # The image is converted to a Tkinter PhotoImage object for display
                photo = ImageTk.PhotoImage(image)
                
            except Exception as e:
                messagebox.showerror("Error", f"Error loading image: {e}")

        if photo:
            img_frame = Frame(self.notebook)
            img_label = Label(img_frame, image=photo)
            img_label.image = photo  # keep a reference to avoid garbage collection
            img_label.pack(fill="both", expand=True)
            
            # The frame with the image is added to the notebook
            self.add_tab(img_frame, "Image")
    
    
    def apply_model(self, model_type):
        """
            Method that applies a machine learning model to previously loaded data.
            It is called when "Logistic Regression", "Decision Tree", "Random Forest", "SVC", "MLP" 
            commands of the "ML Models" Menu are selected.

            :arg: model_type = variable to define the type of ML model to use
        """
        if self.df is not None and not self.df.empty:
            # Card title
            title = f"{model_type.replace('_', ' ').capitalize()}"

            # Check if there is already a card for this type of model
            for tab in self.notebook.tabs():
                if self.notebook.tab(tab, "text").startswith(title):
                    self.notebook.select(tab)
                    return

            X = self.df.iloc[:, :-1]
            y = self.df.iloc[:, -1]
            
            # Split the dataset into training and test sets
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            # A model is selected and created based on the specified type
            if model_type == 'logistic_regression':
                model = LogisticRegression()
            elif model_type == 'decision_tree':
                model = DecisionTreeClassifier()
            elif model_type == 'random_forest':
                model = RandomForestClassifier()
            elif model_type == 'SVC':
                model = SVC()
            elif model_type == 'MLP':
                model = MLPClassifier()
            else:
                messagebox.showerror("Error", "Unknown model type.")
                return

            # The model is trained and used to make predictions on test data
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            
            # Metrics
            accuracy = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted')
            precision = precision_score(y_test, y_pred, average='weighted')
            recall = recall_score(y_test, y_pred, average='weighted')
            
            # Create a Frame to contain the results
            result_frame = Frame(self.notebook)
            result_frame.pack(fill='both', expand=True)
            title_label = Label(result_frame, text=f"{title}:", font=('Courier', 14),fg = "RoyalBlue",
                                anchor='w', justify='left')
            title_label.pack(pady=10, padx = 25, fill = 'x')
            result_label = Label(result_frame, text=f"Accuracy: {accuracy:.2f}", font=('Helvetica', 14))
            result_label.pack(pady=10)

            # Classification report
            report_text = classification_report(y_test, y_pred)
            report_label = Label(result_frame, text=f"Classification Report:\n{report_text}", font=('Courier', 10), justify='left', anchor='w')
            report_label.pack(pady=10)

            # Confusion matrix
            cm = confusion_matrix(y_test, y_pred)
            cm_figure, cm_ax = plt.subplots(figsize=(4, 4))
            cm_ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            cm_ax.grid(False)
            cm_ax.set_xticks([0, 1])
            cm_ax.set_yticks([0, 1])
            cm_ax.set_xticklabels(['Predicted 0s', 'Predicted 1s'])
            cm_ax.set_yticklabels(['Actual 0s', 'Actual 1s'])
            cm_ax.set_ylim(1.5, -0.5)  
            for i in range(2):
                for j in range(2):
                    cm_ax.text(j, i, cm[i, j], ha='center', va='center', color='red')

            cm_canvas = FigureCanvasTkAgg(cm_figure, result_frame)
            cm_canvas.get_tk_widget().pack(fill='both', expand=True)
            

            # Store model results for saving
            self.current_model_results = {
                'model_name': title,
                'accuracy': accuracy,   
                'F1':f1,
                'precision': precision,
                'recall':recall,
                'confusion_matrix': cm,
                'y_pred': y_pred
            }

            # Adding a new tab to the notebook widget with the specified content
            self.add_tab(result_frame, title)

        else:
            messagebox.showerror("Error", "No dataset loaded")
        

    def show_home(self):
        """
            Method for displaying the "Home" tab. 
            This tab serves as a welcome and information point for the user.
            It is called when the "Home" button is pressed.

            :arg:()
        """
        # Check if a Home tab already exists
        for tab in self.notebook.tabs():
            if self.notebook.tab(tab, "text") == "Home":
                self.notebook.select(tab)
                return
        
        # Creating a new frame
        home_frame = Frame(self.notebook)
        # Welcome label
        home_label = Label(home_frame, text="Welcome to the Dataset Manager!", font=('Courier', 18),fg = "RoyalBlue")
        home_label.pack(pady=20)
        
        label1 = Label(home_frame, text="Have fun playing with your dataset:\n model testing, graph visualization and more...", 
                       font=('Courier', 11))
        label1.pack()

        # Upload image
        try:
            image = Image.open("./show.png")  # specify the image path
            image = image.resize((200,200), Image.Resampling.LANCZOS)
            # The image is converted to a Tkinter PhotoImage object for display
            photo = ImageTk.PhotoImage(image)

            img_label = Label(home_frame, image=photo,justify='right')
            img_label.image = photo  # keep a reference to avoid garbage collection
            img_label.pack(pady=20, fill = BOTH, expand=True)
        except Exception as e:
            print(f"Error loading image: {e}")
        
        # Frame to contain the label and the help button
        frame1 = Frame(home_frame)
        frame1.pack(fill ='y')
        label2 = Label(frame1, text="For information on how to use the interface, click here:", 
                       font=('Monaco', 10))
        label2.pack(side='left', padx=10)

        guide_message = "Fast guide:\n1. Upload your dataset\n2. Select a chart to view the data\n3. Test a machine learning model"
        guide_button = Button(frame1, text="Guide", command=lambda:messagebox.showinfo("Guide", guide_message))
        guide_button.pack(side='right')

        # Add program and Python version information
        version_info = f"Program version: {PROGRAM_VERSION}\nPython: {platform.python_version()}"
        version_label = Label(home_frame, text=version_info)
        version_label.pack(side='bottom', pady=10)
        
        # Add home_frame to notebook
        self.add_tab(home_frame, "Home")
   

    def save_content(self):
        """
            Method designed to save the current contents of the application, 
            which can be a graph or the results of a machine learning model.
            It is called when the "Save" command is selected.
            
            :arg:()
        """
        if self.current_plot_image and not self.current_model_results: # if there is a graph and there are no model results
            # Save the graph by opening a dialog box
            file_path = filedialog.asksaveasfilename(defaultextension=".png",
                                                      filetypes=[("PNG files", "*.png"), ("All files", "*.*")])
            if file_path:
                self.current_plot_image.savefig(file_path, format="png")
                messagebox.showinfo("Success", "Plot saved successfully!")

            self.current_plot_image = None  # Reset after saving

        elif self.current_model_results and not self.current_plot_image: # if there are model results and there is no graph
            # Save model results by opening a dialog box
            file_path = filedialog.asksaveasfilename(defaultextension=".txt",
                                                      filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
            if file_path:
                # Writing results to text file
                with open(file_path, 'w') as file: # open for writing
                    # Write the name of the model
                    model_name = self.current_model_results.get('model_name', 'Unknown Model')
                    file.write(f"Model: {model_name}\n\n")

                    # Iterates over the results and writes them to the file
                    for key, value in self.current_model_results.items():
                        if key == 'model_name':
                            continue  # skip typing the model name, already written
                        elif key == 'y_pred':
                            file.write(f"{key}:\n")                           
                            for item in value:
                                # Each element is written to a new line in the file
                                file.write(f"{item}\n")
                        elif key == 'confusion_matrix':
                            file.write(f"{key}:\n")
                            if isinstance(value, (list, np.ndarray)):
                                for row in value:
                                    # Each row of the matrix is ​​transformed into a string of numbers separated by spaces and written to the file
                                    file.write(" ".join(map(str, row)) + "\n")
                            else:
                                file.write(str(value))
                        else:
                            file.write(f"{key}: {value}\n")
                    
                messagebox.showinfo("Success", "Model results saved successfully!")

            self.current_model_results = None  # Reset after saving
        else:
            messagebox.showwarning("Warning", "No plot or model results to save.")
            
            
    def add_tab(self, frame, title):
        """
            Method that adds a new tab to the notebook
            
            :arg: frame = frame to add as a new tab
                  title = tab name
        """
        self.notebook.add(frame, text=title)
        self.notebook.select(frame) # select the newly added tab making it active
        # Adds a close button to the tab
        self.add_close_button(frame)


    def add_close_button(self, frame):
        """
            Method that adds a close button to a tab of the notebook
            
            :arg: frame = the frame to which to add the close button
        """
        # Gets the index to identify the card
        tab_id = self.notebook.index(frame)
        # Update the card text without changing it
        self.notebook.tab(tab_id, text=f"{self.notebook.tab(tab_id, 'text')}") # useful for ensuring that visual changes are applied correctly
        # Creating a closing button
        close_button = Button(frame, text="x", command=lambda: self.notebook.forget(frame), padx=2, pady=2)
        close_button.place(x=0, y=0, anchor="nw") # position of the botton

   


if __name__ == "__main__":
    root = Tk() # main window
    root.title("Dataset Manager")

    # Geometry
    root.geometry('800x500') 
    root.iconbitmap('./machine-learning.ico')

    # Creating an instance of the class Datawindow
    app = Datawindow(root)

    root.mainloop() 